In [ ]:
######## Importation des bibliothèques

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
######## échauffement : le broadcasting

a = np.array([1,2,3])

#1
print(np.shape(a[:,None]),a[:,None]) #on rajoute une dimension a 'droite' --> vecteur colonne (3,1)
print(np.shape(a[None,:]),a[None,:]) #on rajoute une dimension a 'gauche' --> un tableau dans un tableau (1,3)

#2
print(np.shape(a[:,None]-a[None,:]),a[:,None]-a[None,:]) #on etend la première matrice pr qu'elle ait 3 colonnes, on etend la seconde pour qu'elle ai 3 lignes

#3
b = np.arange(1,7).reshape(2,3)
print(b[:,None,:]) #on crée des matrices autour de celle déja existantes (des matrices de matrices)
print(b[:,:,None]) #on crée 2 matrices avec 3 matrices dedans (les plus petites matrices ont 1 élément)
print(np.shape(b[:,None,:]-b[::,None]),b[:,None,:]-b[:,:,None]) 


(3, 1) [[1]
 [2]
 [3]]
(1, 3) [[1 2 3]]
(3, 3) [[ 0 -1 -2]
 [ 1  0 -1]
 [ 2  1  0]]
[[[1 2 3]]

 [[4 5 6]]]
[[[1]
  [2]
  [3]]

 [[4]
  [5]
  [6]]]
(2, 1, 3) [[[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]

 [[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]]


In [ ]:
######## initialiation aléatoire
# les borens pour le tirage au sort initial
mass_max=3

x_min, x_max = -10., 10.
y_min, y_max = -10., 10.

speed_max = 1

def init_problem(N):
    masses = np.random.uniform(0.01,mass_max,N)
    minimums = np.array([[x_min], [y_min]]) # on veut que la première ligne soit en x et la seconde en y, par broadcasting ça va s'étandre sur les N colonnes
    maximums = np.array([[x_max],[y_max]])
    positions = np.random.uniform(minimums, maximums, (2,N))
    speeds = np.random.uniform(0.01,speed_max,(2,N)) #on cree un tableau de vx et vy en broadcastant les v_min et v_max
    return masses, positions, speeds

masses, positions, speeds = init_problem(10)
try:
    assert masses.shape == (10,) and positions.shape == speeds.shape == (2,10) 
    print("OK")
except:
    print("KO")

OK


In [ ]:
######## initialisation reproductible

def init3():
    # first element is sun-like: heavy, at the center, and no speed
    masses = np.array([3, 1, 1], dtype=float)
    positions = np.array([
        [0, 5, -5], 
        [0, 1, -1]], dtype=float)
    speeds = np.array([
        [0, -1, 1], 
        [0, 0, 0]], dtype=float)
    return masses, positions, speeds

def forces(masses, positions, G=1.0):
    diff = positions[:,:,None]-positions[:,None,:] # on crée des tableaux undividuels des positions (rajout de la 3ème dimension) pour qu'elles soient broadcaster 
    # sur toutes les autres positions. on rajoute la deuxième dimension pour que chaqune des liste de position soit broadcaster sur chacune des positions individuelles
    distance = np.linalg.norm(diff,axis=2) # on fait la norme euvlidienne des éléments entre eux dans la 3ème dimension
    np.fill_diagonal(distance, np.inf) # on remplace les element quand i=j par infini, comme c'est au dénominatuer la force sera nulle
    coeff = masses[:,None]*masses[None,:] # On crée des tableaux individuel(rajout de la dimension a droite) pour que chaque masse soit broadcaster sur l'entierete des autres masses (on broadcast chaque tableau de masse aussi)
    produit = G*coeff*diff/(distance**3) # on fait le produit, il n'y a pas de problème de dimension
    somme = np.sum(produit,axis=2) # on ecrase le tableau dans le sens de la troisième dimension lors de la somme
    return somme


In [ ]:
######## Le simulateur

def simulate(masses, positions, speeds, dt=0.1, nb_steps=100):
    N = len(masses)
    resultat=np.zeros(nb_steps,len(positions),N) # on crée le tableau ou on mettra la succession de coordonnées
    m=masses[None,:]
    resultat[0]=positions # on initie les positions des particules
    for i in range(np_steps):
        F=forces(masses,positions) 
        a=F/m # on calcul l'accélération de chaque particules
        speeds=speeds+a*dt # a chaque itération on calcul la vitesse et les position au temps dt suivant
        positions=positions+speeds*dt
        resultat[i]=positions
    return resultat

In [ ]:
######## dessiner
def draw(simulation, masses, colors=None, scale=10):
    figure,ax=plt.subplots()
    for i in range(len(masses)):
        X,Y=simulation[:,:,i].T[0],simulation[:,:,i].T[1]
        if colors is not None:
            print(colors[i])
            plt.plot(X,Y, color=colors[i])
            ax.scatter(X[-1],Y[-1],s=30*masses[i], color=colors[i])
        else:
            plt.plot(X,Y)
            ax.scatter(X[-1],Y[-1],s=30*masses[i])
    plt.show()

draw(simulate(init3()[0],init3()[1],init3()[2]),init3()[0])